# 4 Searching architectures instead of guessing them

A training run now costs about two seconds. That single fact makes hand-tuning
indefensible — not inefficient, *indefensible*, because you can no longer claim you picked
the width by intuition when checking twenty widths would have cost a coffee.

So we stop guessing. This notebook hands the architecture space to a search algorithm and
reads what comes back, honestly, including the parts that are disappointing.

**We switch datasets here, on purpose.** Notebooks 01–03 used EuroSAT, where the point was
that the approach works at all. But EuroSAT's three encoders land within three points of
each other, so a search there is flat on every axis — including the one that is supposed
to matter. DTD (textures) spreads those same encoders across fourteen points and leaves
about twenty points of headroom. Run your search where the result is legible.

In [ ]:
import os
import warnings

# ray propagates `uv run` to its workers, which then expect pyproject.toml in the notebook's
# own directory. It is one level up, so turn that off. Must happen before `import ray`.
os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import sys
import time
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import Concatenate2D, GaussianNoise, NeuralNet, Serial
from vectormesh.components.metrics import Accuracy
from vectormesh.data import Collate, CollateParallel, OneHot
from vectormesh.training import Trainer, TrainerSettings

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

EPOCHS = param(15, test=1)
N_TRIALS = param(24, test=2)
SEEDS = param(5, test=2)

ENCODERS = ["dinov2-large", "dinov2-base", "dinov2-small", "resnet-18"]
caches = {
    enc: {split: VectorCache.from_hub(f"pttrn-io/dtd-{enc}", split=split) for split in ("train", "test")}
    for enc in ENCODERS
}

N_CLASSES = caches["dinov2-small"]["train"].num_classes
DIM = {enc: caches[enc]["train"].dim for enc in ENCODERS}

print(f"DTD: {len(caches['dinov2-small']['train'])} train / {len(caches['dinov2-small']['test'])} test, {N_CLASSES} classes")
print("encoder widths:", DIM)

In [ ]:
from typing import Optional
DEFAULT_ONEHOT = OneHot(num_classes=N_CLASSES, label_col="label", target_col="onehot")
DEFAULT_COLLATE = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)


def loaders(encoder: str, batch_size: int = 64, onehot: Optional[OneHot] = None, collate: Optional[Collate] = None):
    # onehot/collate are arguments, not a closure over a module global -- loaders(name) still
    # works via the defaults below, but a caller with a different label scheme is not stuck.
    onehot = onehot or DEFAULT_ONEHOT
    collate = collate or DEFAULT_COLLATE
    return [
        DataLoader(caches[encoder][split].dataset.map(onehot), batch_size=batch_size,
                   shuffle=(split == "train"), collate_fn=collate)
        for split in ("train", "test")
    ]


class TimedTrainer(Trainer):
    """A Trainer that adds up the seconds spent in training steps; evaluation after every
    epoch is not counted."""

    train_seconds = 0.0

    def trainbatches(self) -> float:
        start = time.perf_counter()
        loss = super().trainbatches()
        self.train_seconds += time.perf_counter() - start
        return loss


def run(model, train_dl, test_dl, lr: float = 1e-3, epochs: int = EPOCHS) -> tuple[float, float]:
    """Train `model` and return (test accuracy, seconds spent in training steps)."""
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(train_dl),
        valid_steps=len(test_dl),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = TimedTrainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=train_dl,
        validdataloader=test_dl,
        progress=False,
    )
    return trainer.loop().metric_dict["Accuracy"], trainer.train_seconds

## 4.0 The baseline, before anything else

Notebook 03 said it and it is worth repeating as a rule: **a search without a baseline is
unreadable.** If you do not know what the two-line model scores, you cannot tell whether
the search's best trial is a discovery or a rounding error.

So establish it first, in two steps. Start with a floor: 1-nearest-neighbour, which trains
nothing at all — every test vector takes the label of its closest training vector. Then
the two-line model itself, with a seed spread, because a single run is an anecdote.

### Every run goes to MLflow, starting now

Every number this notebook produces — the floor, the baseline, each search trial, the
fusion arm — is logged as an MLflow run, all in one experiment called `lesson1-search`.
The MLflow UI then shows them as rows of one table, sortable on `accuracy`, and the
question "did the search beat the baseline?" becomes a sort, not a scroll back through
cell outputs.

The logging goes through `log_run` from `runlog.py`, the same helper notebook 05 uses, so
every run has the same shape:

- a **`role`** tag: `baseline`, `trial` (one search trial), `hypothesis` (the fusion arm),
  or `ablation` (the 1-NN floor: the baseline with the training taken out);
- the **parameters** that define it (`encoder`, `head`, `n_hidden_layers`, `hidden_dim`,
  `sigma`, `lr`, `epochs` — whichever apply);
- **`accuracy`**, and when a run was repeated over seeds also `accuracy_std`,
  `accuracy_per_seed` and `n_seeds`;
- the **architecture**: `n_params`, a one-line `architecture` tag, and an
  `architecture.json` artifact with the tensor shape after every layer.

Two extra tags, `notebook` and `session` (the time you started this run of the notebook),
keep a rerun from getting mixed up with the last one.

In [ ]:
from runlog import log_run, use_experiment

TRACKING_URI = use_experiment("lesson1-search")
TAGS = {"notebook": "04_search", "session": time.strftime("%Y-%m-%d %H:%M:%S")}

print(f"tracking to {TRACKING_URI}")
print(f"experiment 'lesson1-search', session tag '{TAGS['session']}'")

In [ ]:
MODEL = "dinov2-small"  # the encoder the fusion arm in 4.3 is measured against
print(f"floor: {MODEL} -> {N_CLASSES} classes, {DIM[MODEL]} features")
tr = caches[MODEL]["train"]
te = caches[MODEL]["test"]

# cache["col"] is a lazy Column; the [:] materialises it into one tensor
Xtr, ytr = tr["embed"][:], tr["label"][:]
Xte, yte = te["embed"][:], te["label"][:]

start = time.time()
A = torch.nn.functional.normalize(Xtr, dim=1)     # cosine similarity = dot product
B = torch.nn.functional.normalize(Xte, dim=1)     # once both sides are unit length
neighbour = (B @ A.T).argmax(dim=1)               # (1000, 4000) -> index of the closest
accuracy = (ytr[neighbour] == yte).float().mean()

print(f"train {tuple(Xtr.shape)}   test {tuple(Xte.shape)}")
print(f"took {1000 * (time.time() - start):.0f} ms, trained 0 parameters")
print(f"chance level: {1 / N_CLASSES:.4f}\n")
print(f"1-nearest-neighbour on {N_CLASSES} classes: {accuracy:.4f}")

KNN_FLOOR = float(accuracy)
log_run("1nn-floor", "ablation", dataset="dtd",
        params={"dataset": "dtd", "encoder": MODEL, "embed_dim": DIM[MODEL], "head": "1-nearest-neighbour"},
        scores={"accuracy": [KNN_FLOOR]}, train_seconds=0.0, tags=TAGS)

In [ ]:
train_dl, test_dl = loaders(MODEL)

scores, seconds = [], []
start = time.time()
for seed in range(SEEDS):
    torch.manual_seed(seed)
    baseline = torch.nn.Linear(DIM[MODEL], N_CLASSES)
    accuracy, spent = run(baseline, train_dl, test_dl)
    scores.append(accuracy)
    seconds.append(spent)

BASELINE = float(np.mean(scores))
BASELINE_STD = float(np.std(scores))

# one run for all seeds: the score is the mean, the spread is logged next to it
log_run("baseline", "baseline", dataset="dtd",
        params={"dataset": "dtd", "encoder": MODEL, "embed_dim": DIM[MODEL], "head": "linear",
                "n_hidden_layers": 0, "sigma": 0.0, "lr": 1e-3, "epochs": EPOCHS},
        scores={"accuracy": scores}, train_seconds=float(np.mean(seconds)),
        model=baseline, example=torch.zeros(2, DIM[MODEL]), tags=TAGS)

print(f"seeds: {[f'{s:.4f}' for s in scores]}")
print(f"\nBASELINE  {BASELINE:.4f} +/- {BASELINE_STD:.4f}   ({time.time() - start:.0f}s for {SEEDS} runs)")
print(f"anything inside +/-{2 * BASELINE_STD:.4f} of that is not a result.")
print(f"\ntraining over the 1-NN floor ({KNN_FLOOR:.4f}): {100 * (BASELINE - KNN_FLOOR):+.1f} pt")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 1.9))
ax.axvspan(BASELINE - 2 * BASELINE_STD, BASELINE + 2 * BASELINE_STD, color="tab:blue", alpha=0.12,
           label="baseline mean +/- 2 std: not a result")
ax.scatter(scores, np.zeros(len(scores)), color="tab:blue", zorder=3, label=f"baseline, one dot per seed ({SEEDS})")
ax.axvline(BASELINE, color="tab:blue", lw=1)
ax.scatter([KNN_FLOOR], [0], marker="D", color="k", zorder=3, label="1-NN floor, nothing trained")
ax.set_yticks([])
ax.set_xlabel("test accuracy")
ax.set_xlim(min(KNN_FLOOR, min(scores)) - 0.02, max(scores) + 0.02)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8, frameon=False)
ax.set_title(f"the reference every later number is read against   (chance: {1 / N_CLASSES:.3f})", fontsize=10, loc="left")
plt.tight_layout()
plt.show()

The spread is the number people skip, and it is the one that decides what counts as a
win. Write both down. A trial that beats the mean by less than the spread has not beaten
anything.

**Open MLflow now**, before the search adds two dozen more rows. In a terminal, from
`notebooks/1_pytorch_intro`:

```bash
uv run mlflow ui --backend-store-uri sqlite:///../../artefacts/mlflow/mlruns.db
```

and open <http://127.0.0.1:5000>. Pick the experiment `lesson1-search`. You should see a
`1nn-floor` run and a `baseline` run, each with an `accuracy` metric; the baseline also
has `accuracy_std` and one `accuracy_per_seed` value per seed. Click the baseline run and
open its Artifacts tab: `architecture.json` lists every layer with its input and output
shape. Typing `tags.role = "baseline"` in the search box shows only baselines; from §4.1
on the search trials land in this same table, with `tags.role = "trial"`.

## 4.1 The search space

One dictionary describes every model we are willing to consider:

- **`encoder`** — which cache to train on. Notebook 01 measured this axis at fourteen
  points on this dataset, and it is the reason the search is worth running. It sets `D`,
  the width of the vectors going in.
- **`n_hidden_layers`** — `0`, `1` or `2`. `0` is the baseline from §4.0, one
  `Linear(D, 47)`. `1` and `2` put that many hidden layers of width `hidden_dim` in front
  of it, with a GELU after each. vectormesh's `NeuralNet` builds that chain in one call:
  `NeuralNet([D, H, H], 47)` is `Linear(D, H) → GELU → Linear(H, H) → GELU → Linear(H, 47)`.
  One `NeuralNet` with the whole width chain, not two `NeuralNet`s stacked: stacking puts
  a bare `Linear → Linear` seam in the middle, notebook 02's "depth without nonlinearity"
  collapse. (A one-element list, `NeuralNet([D], 47)`, still has one hidden layer of width
  `D`, so the `0` case is a plain `nn.Linear`.)
- **`hidden_dim`** — `H`, the width of each hidden layer. Meaningless when
  `n_hidden_layers = 0`, and the model ignores it in that case.
- **`sigma`** — how much Gaussian noise to add to the input vectors during training. Pixel
  augmentation is impossible now (the pixels are gone), so this is the feature-space
  version: it teaches the head not to trust any single dimension too much.
- **`lr`** — sampled log-uniformly, because learning rates matter multiplicatively.

`build_model` turns one such configuration into a model. The search's trials use it, and
so do the diagrams below, so what you see drawn is what gets trained.

In [ ]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch


def build_model(config: dict, dim: int, n_classes: int) -> nn.Module:
    n_hidden = config["n_hidden_layers"]
    if n_hidden == 0:
        head = nn.Linear(dim, n_classes)                                      # the baseline
    else:
        head = NeuralNet([dim, *[config["hidden_dim"]] * n_hidden], n_classes)  # GELU between every Linear
    return Serial([GaussianNoise(sigma=config["sigma"]), head])


param_space = {
    "encoder": tune.choice(ENCODERS),
    "n_hidden_layers": tune.choice([0, 1, 2]),   # 0 = the baseline's single Linear
    "hidden_dim": tune.choice([128, 512]),       # ignored when n_hidden_layers = 0
    "sigma": tune.uniform(0.0, 0.4),
    "lr": tune.loguniform(1e-4, 3e-3),
    "epochs": EPOCHS,
}

### What each option changes in the model

Every model in the space has the same skeleton: a batch of `B` cached vectors of width `D`
goes in, noise is added, a head maps it to 47 class scores, and the loss compares those
with the labels. The drawing below is made by `trace`, which pushes a dummy batch through
the real `build_model(...)` output and writes the shape of every tensor on the arrow it
travels along. Read the shapes: the batch dimension `B` never changes, the noise leaves the
shape alone, and every `Linear(a → b)` turns `(B, a)` into `(B, b)`.

In [ ]:
from diagrams import Block, draw_architectures, plot_search_space, sample_config, trace

LOSS = Block("CrossEntropyLoss\nvs labels", kind="loss")


def diagram(config: dict) -> list:
    """Input cache, the traced model, the loss: one row of a drawing."""
    encoder, dim = config["encoder"], DIM[config["encoder"]]
    model = build_model(config, dim, N_CLASSES)
    return [Block(f"{encoder}\ncache", f"(B, {dim})", "data"), *trace(model, torch.zeros(2, dim)), LOSS]


example = {"encoder": "dinov2-small", "hidden_dim": 512, "sigma": 0.1}
rows = {f"n_hidden_layers = {n}": diagram({**example, "n_hidden_layers": n}) for n in (0, 1, 2)}
note = "\n".join([
    "encoder      sets D, the input width: " + ", ".join(f"{e} {d}" for e, d in DIM.items()),
    "hidden_dim   sets H, the width of every hidden layer: 128 or 512 (drawn: 512)",
    "sigma        sets the noise strength; at test time the noise block passes its input through unchanged",
    "lr           is not in the drawing: it sets the optimizer's step size, not the model",
])
draw_architectures(rows, note=note)
plt.show()

### What the search will sample

`param_space` is not a list of models, it is a set of distributions, one per axis. A trial
draws one value from each. Here are a few thousand draws per axis, before a single model
is trained, with one trial's draw marked in red:

- the three `choice` axes are flat: every option is equally likely;
- `sigma` is flat between 0 and 0.4;
- `lr` is flat **on a log axis**: as many trials between 1e-4 and 3e-4 as between 1e-3
  and 3e-3. The last panel shows the same draws on a linear axis, where they pile up at
  the low end. That is the point: a step from 1e-4 to 2e-4 doubles the learning rate, a
  step from 2.9e-3 to 3e-3 barely changes it, so the small values deserve as many trials.

Below the panels is the model that one red trial would train.

In [ ]:
trial = sample_config(param_space, seed=6)
print("one trial:", {k: round(v, 5) if isinstance(v, float) else v for k, v in trial.items()})

plot_search_space(param_space, trial=trial)
plt.show()

draw_architectures({"the model this trial trains": diagram(trial)})
plt.show()

We use **TPE** (`HyperOptSearch`) to pick the trials. The first few are drawn from the
distributions above; after that TPE models which regions produced good scores and samples
where the expected improvement is highest. A grid search would spend most of its budget on
combinations it already knows are bad. Lesson 3 opens up the algorithms themselves; here it
is one tool, used properly.

**On `hidden_dim` being meaningless at `n_hidden_layers = 0`:** Ray *can* express a
genuinely conditional space — `HyperOptSearch(space=...)` accepts a native hyperopt space
built with `hp.choice`, which lets `hidden_dim` live only inside the `n_hidden_layers > 0`
branch, so TPE never wastes a sample point on a combination that cannot exist. That is
worth knowing, and worth reaching for once a search space has real nesting. It is not worth
building here: two flat axes, with the unused one simply ignored by `build_model`, cost
nothing at this trial budget. Reach for a hand-built hyperopt space when flat axes start
wasting a noticeable fraction of the budget, not by default.

In [ ]:
def trainable(config):
    # ray runs this in its own process, so everything it needs is set up inside -- the
    # logger too: it is imported here because the notebook's logger cannot be pickled
    from loguru import logger
    logger.remove()
    logger.add(sys.stderr, level="WARNING")
    encoder = config["encoder"]
    train = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="train")
    test = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="test")

    oh = OneHot(num_classes=train.num_classes, label_col="label", target_col="onehot")
    col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
    tr = DataLoader(train.dataset.map(oh), batch_size=64, shuffle=True, collate_fn=col)
    te = DataLoader(test.dataset.map(oh), batch_size=64, shuffle=False, collate_fn=col)

    torch.manual_seed(0)
    model = build_model(config, train.dim, train.num_classes)
    settings = TrainerSettings(
        epochs=config["epochs"],
        metrics=[Accuracy()],
        logdir=Path("ray_results/trials").resolve(),  # never created unless earlystop saves
        train_steps=len(tr),
        valid_steps=len(te),
        optimizer_kwargs={"lr": config["lr"]},
        earlystop_kwargs=None,
    )
    trainer = TimedTrainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=tr,
        validdataloader=te,
        progress=False,  # ray's own status table already shows progress across trials
    )
    result = trainer.loop()
    tune.report({"Accuracy": result.metric_dict["Accuracy"], "train_seconds": trainer.train_seconds})

In [ ]:
tuner = tune.Tuner(
    trainable,
    param_space=param_space,
    tune_config=tune.TuneConfig(
        metric="Accuracy",
        mode="max",
        search_alg=HyperOptSearch(),
        num_samples=N_TRIALS,
    ),
    run_config=tune.RunConfig(
        storage_path=Path("ray_results").resolve(),
        name="dtd_head_search",
        verbose=1,  # in Jupyter, ray renders a live, self-updating status table of the trials
    ),
)

start = time.time()
results = tuner.fit()
print(f"{N_TRIALS} trials in {time.time() - start:.0f}s")

Each trial now becomes an MLflow run, in the same experiment as the floor and the baseline.
The logging happens here, after the search, from this notebook's own process: every ray
trial runs in a separate process, and several processes writing to one SQLite file at the
same time is asking for a locked database. Each trial's run records its configuration, its
accuracy and the architecture it trained.

In [ ]:
for i, result in enumerate(results):
    config = dict(result.config)
    dim = DIM[config["encoder"]]
    head = "linear" if config["n_hidden_layers"] == 0 else "mlp"
    log_run(f"trial-{i:02d}", "trial", dataset="dtd",
            params={**config, "dataset": "dtd", "embed_dim": dim, "head": head},
            scores={"accuracy": [result.metrics["Accuracy"]]},
            train_seconds=result.metrics["train_seconds"],
            model=build_model(config, dim, N_CLASSES), example=torch.zeros(2, dim), tags=TAGS)

print(f"logged {len(results)} trials to 'lesson1-search'. In the MLflow UI, sort on accuracy: "
      f"where does the baseline land between the trials?")

## 4.2 Reading the result

The temptation is to read off the winning config and move on. Do the opposite: look at how
accuracy varies along **each axis separately**, which tells you whether the axis mattered
at all.

In [ ]:
df = results.get_dataframe()
df = df[["Accuracy", "config/encoder", "config/n_hidden_layers", "config/hidden_dim", "config/sigma", "config/lr"]]
df.columns = ["accuracy", "encoder", "n_hidden_layers", "hidden_dim", "sigma", "lr"]

print(df.sort_values("accuracy", ascending=False).head(8).to_string(index=False))

print(f"\n{'encoder':<14} {'trials':>7} {'worst':>8} {'best':>8}")
print("-" * 40)
for enc, group in df.groupby("encoder"):
    print(f"{enc:<14} {len(group):>7} {group['accuracy'].min():>8.4f} {group['accuracy'].max():>8.4f}")

dino = df[df.encoder.str.startswith("dinov2")]["accuracy"]
resnet = df[df.encoder == "resnet-18"]["accuracy"]
best_row = df.loc[df["accuracy"].idxmax()]
if len(dino) and len(resnet):
    print(f"\ngap between the worst dinov2 trial and the best resnet trial: {100 * (dino.min() - resnet.max()):+.1f} pt")
print(f"\nbaseline (no search, {SEEDS} seeds): {BASELINE:.4f} +/- {BASELINE_STD:.4f}")
print(f"best trial found by the search    : {best_row['accuracy']:.4f}  ({best_row['encoder']})")
print(f"n_hidden_layers of the best trial  : {int(best_row['n_hidden_layers'])}  (0 = the baseline's single Linear)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
order = {"encoder": ENCODERS, "n_hidden_layers": [0, 1, 2]}

for ax, col in zip(axes, ["encoder", "n_hidden_layers", "sigma", "lr"]):
    if col in order:
        # categories at fixed positions, in a fixed order, whatever order the trials came in
        position = {value: i for i, value in enumerate(order[col])}
        ax.scatter(df[col].map(position), df["accuracy"], alpha=0.7)
        for value, group in df.groupby(col)["accuracy"]:
            ax.plot([position[value]], [group.mean()], marker="_", ms=40, color="crimson", mew=2)
        ax.set_xticks(range(len(order[col])), [str(v) for v in order[col]])
        ax.set_xlim(-0.5, len(order[col]) - 0.5)
    else:
        ax.scatter(df[col], df["accuracy"], alpha=0.7)
        if col == "lr":
            ax.set_xscale("log")
            ax.tick_params(axis="x", which="minor", labelbottom=False)  # decades only
    ax.axhline(BASELINE, color="k", ls="--", lw=1)
    ax.axhspan(BASELINE - BASELINE_STD, BASELINE + BASELINE_STD, color="k", alpha=0.08)
    ax.set_xlabel(col)
    ax.tick_params(axis="x", rotation=30 if col == "encoder" else 0)

axes[0].set_ylabel("test accuracy")
fig.suptitle("one axis at a time; red = mean per option, dashed line = the baseline (one Linear on dinov2-small), band = its seed spread")
plt.tight_layout()
plt.show()

In [ ]:
from plotting import search_parallel_plot

# The per-axis panels above are good for "does this axis matter at all" -- they are bad at
# showing INTERACTIONS, e.g. "does hidden_dim matter only when n_hidden_layers > 0". A
# parallel-categories plot draws every trial as one line across all axes at once, so a
# pattern that only shows up jointly is still visible. Categorical config values (like
# encoder) plot natively -- no manual code/tickvals mapping needed.
fig = search_parallel_plot(df, columns=["encoder", "n_hidden_layers", "hidden_dim", "sigma", "lr", "accuracy"])
fig.update_layout(title="Every trial, every axis at once -- colour is accuracy")
fig.show()

Read the four per-axis panels left to right; the parallel-coordinates plot then shows the
same trials as one picture instead of four.

**The encoder panel separates into bands.** The gap printed above is the headline: the
*worst* DINOv2 trial in the search beats the *best* resnet-18 trial by more than ten
points, and nothing done to the head closes it. Among the DINOv2 sizes, `dinov2-base` and
`dinov2-large` sit a few points above `dinov2-small`.

**`sigma` and `lr` are clouds.** They scatter trials up and down *within* a band without
organising them. Inside a band much of the visible variation is the seed noise you
measured in §4.0 — each trial here is a single seed.

**`n_hidden_layers` moves less than it looks.** This panel mixes all four encoders, so
the mean of a column says as much about which encoders TPE happened to try there as about
depth. Compare within one encoder on the leaderboard instead. On `dinov2-small` a hidden
layer is worth about two points over a single `Linear` (over five seeds, 0.81 against the
baseline's 0.79); on the larger DINOv2 encoders the best trials turn up with and without
one. In the parallel-coordinates plot: the brightest lines run through `dinov2-base` and
`dinov2-large`, then fan out across `n_hidden_layers`, `hidden_dim`, `sigma` and `lr`.

Put the two effects side by side and the answer to "what should I spend my time on" is
plain:

> **The leverage is upstream, in the encoder.** Choosing the vectors moved accuracy by
> more than ten points; everything between the vectors and the loss — the part that feels
> most like "doing machine learning" — moved it by about two.

### Do not over-generalise this

This holds for **a small head on a strong frozen encoder**. It is not a law about neural
architecture, and two later lessons are counter-examples by construction:

- **Lesson 3 (time series)** has no general-purpose encoder worth freezing. You window the
  series and train end to end, and the architecture is back to being your problem.
- **Lesson 6 (self-supervised)** is about *making* the representation instead of renting
  one. That is precisely the place where architecture decides everything.

The transferable lesson is not "architecture never matters". It is: **find out where the
leverage is on your problem before spending your week on the part that feels most like
engineering.**

## 4.3 The fusion arm

Here is an idea with a good story attached. We have two encoders. DINOv2 is strong, resnet
is weak — but they were trained differently, so perhaps they made *different* mistakes. If
resnet knows anything DINOv2 does not, a model that sees both should beat either alone.

That is a real hypothesis, it is cheap to test, and this is how it is built: `join` the two
caches on `source_idx`, so every row carries both vectors of the same image, and collate
two columns instead of one. Each batch is then a *pair* of tensors, `(B, 384)` and
`(B, 512)`, and the model has to turn that pair into one `(B, 47)`.

**Before you run the cells after the drawing, commit to a prediction.** Write it down —
sign and rough size. Does fusing a strong and a weak encoder beat the strong one alone,
and by how much?

A prediction you got wrong teaches you more than one you were handed.

In [ ]:
def joined_loaders(batch_size: int = 64):
    out = []
    for split in ("train", "test"):
        # join() aligns rows on source_idx and raises if the two caches disagree
        joined = caches["dinov2-small"][split].join(caches["resnet-18"][split], on="source_idx")

        col2 = next(c for c in joined.column_names if c.startswith("embed_"))
        collate2 = CollateParallel(vec1_col="embed", vec2_col=col2,
                                   target_col="onehot", padder=torch.stack)
        out.append(DataLoader(joined.dataset.map(DEFAULT_ONEHOT), batch_size=batch_size,
                              shuffle=(split == "train"), collate_fn=collate2))
    return out


fuse_train, fuse_test = joined_loaders()
(X1, X2), y = next(iter(fuse_train))
print(f"two streams per item: {tuple(X1.shape)} and {tuple(X2.shape)}  ->  target {tuple(y.shape)}")


def make_fusion():
    d1, d2 = DIM["dinov2-small"], DIM["resnet-18"]
    return Serial([
        Concatenate2D(),                     # (B, 384) + (B, 512) -> (B, 896)
        nn.Linear(d1 + d2, N_CLASSES),       # the same single Linear as the baseline, wider
    ])

Two inputs means two branches, and the one design decision is **how the branches are
joined**. The drawing shows the two usual answers, with the shape on every arrow:

- **concatenate** (top, what `make_fusion` builds and what is drawn from the real model):
  the widths add up, `(B, 384)` and `(B, 512)` become `(B, 896)`. Nothing is lost and the
  widths do not have to match; the `Linear` after it decides how much to use of each.
- **sum** (bottom, drawn for comparison, not trained here): `a + b` only exists when both
  tensors have the same shape, so one branch first needs a `Linear` to bring 512 down to
  384. In vectormesh that per-branch step is `Parallel([nn.Identity(), nn.Linear(512, 384)])`:
  `Parallel` applies one module to each tensor of the pair. Sum is cheaper after the join,
  but it forces both encoders into one shared 384-wide space, and the projection is one
  more thing to train.

In [ ]:
d1, d2 = DIM["dinov2-small"], DIM["resnet-18"]
caches_in = [[Block(f"dinov2-small\ncache", f"(B, {d1})", "data")],
             [Block(f"resnet-18\ncache", f"(B, {d2})", "data")]]

rows = {
    "concatenate: the widths add up  (make_fusion, traced)": [
        caches_in, *trace(make_fusion(), (X1[:2], X2[:2])), LOSS,
    ],
    f"sum: the widths must match, so resnet's branch is projected to {d1} first": [
        [[Block("dinov2-small\ncache", f"(B, {d1})", "data")],
         [Block("resnet-18\ncache", f"(B, {d2})", "data"), Block(f"Linear\n{d2} → {d1}", f"(B, {d1})", "proj")]],
        Block("a + b", f"(B, {d1})", "join"),
        Block(f"Linear\n{d1} → {N_CLASSES}", f"(B, {N_CLASSES})"),
        LOSS,
    ],
}
draw_architectures(rows)
plt.show()

`Concatenate2D` takes the pair directly, at the two streams' own widths. There is no
`Parallel` in front of it because neither stream gets a transform of its own before the
join; `Parallel` earns its place only when the branches *do* something, as in the sum
variant above. The head after the join is the same single `Linear` as the baseline, so the
second stream is the only thing that differs from §4.0.

Now run it. **One seed first**, the way you would if you were in a hurry.

In [ ]:
torch.manual_seed(0)
one_seed, _ = run(make_fusion(), fuse_train, fuse_test)

print(f"fusion, seed 0           : {one_seed:.4f}")
print(f"baseline, seed 0         : {scores[0]:.4f}")
print(f"baseline, mean of {SEEDS} seeds: {BASELINE:.4f}")
print(f"difference with the mean : {100 * (one_seed - BASELINE):+.2f} pt")

Whatever sign that came out, **it is not yet a result.** You have one number from one seed
against a mean of several. §4.0 measured how much this task wobbles between seeds; a
single run cannot see past that.

So do it properly.

In [ ]:
def seed_spread(make_model, train_dl, test_dl, seeds: int = SEEDS) -> tuple[np.ndarray, float]:
    """Accuracy per seed, and the mean seconds spent in training steps."""
    out, seconds = [], []
    for seed in range(seeds):
        torch.manual_seed(seed)
        accuracy, spent = run(make_model(), train_dl, test_dl)
        out.append(accuracy)
        seconds.append(spent)
    return np.array(out), float(np.mean(seconds))


single = np.array(scores)                                   # the baseline runs from 4.0
fusion, fusion_seconds = seed_spread(make_fusion, fuse_train, fuse_test)
log_run("fusion", "hypothesis", dataset="dtd",
        params={"dataset": "dtd", "encoder": "dinov2-small+resnet-18",
                "embed_dim": DIM["dinov2-small"] + DIM["resnet-18"], "head": "concat+linear",
                "n_hidden_layers": 0, "sigma": 0.0, "lr": 1e-3, "epochs": EPOCHS},
        scores={"accuracy": fusion}, train_seconds=fusion_seconds,
        model=make_fusion(), example=(X1[:2], X2[:2]), tags=TAGS)

n_params = lambda m: sum(p.numel() for p in m.parameters())
diff = fusion.mean() - single.mean()
pooled = np.sqrt(fusion.var(ddof=1) / len(fusion) + single.var(ddof=1) / len(single))

print(f"{'setup':<22} {'mean':>8} {'std':>8} {'params':>10}")
print("-" * 52)
print(f"{'baseline (dinov2)':<22} {single.mean():>8.4f} {single.std():>8.4f} "
      f"{n_params(nn.Linear(DIM['dinov2-small'], N_CLASSES)):>10,}")
print(f"{'fusion (both)':<22} {fusion.mean():>8.4f} {fusion.std():>8.4f} {n_params(make_fusion()):>10,}")
print()
print(f"difference      : {100 * diff:+.2f} pt")
print(f"in sigma        : {diff / pooled:+.1f}")
print(f"fusion wins in  : {int((fusion > single.mean()).sum())}/{len(fusion)} seeds")
print(f"variance cost   : {fusion.std() / single.std():.1f}x the baseline's seed spread")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
setups = ["baseline\n(dinov2)", "fusion\n(both)"]
means = [single.mean(), fusion.mean()]
stds = [single.std(), fusion.std()]

ax.bar(setups, means, yerr=stds, capsize=8, color=["tab:blue", "tab:orange"])
ax.set_ylabel("test accuracy")
ax.set_ylim(min(means) - max(stds) - 0.03, max(means) + max(stds) + 0.03)
ax.set_title(f"mean ± std over {len(single)} seeds")
plt.tight_layout()
plt.show()

### What the table says

Read the table above, not your hopes. On this run, over enough seeds:

**Fusion beats the baseline, and by more than the noise.** About two points, on every
seed, with a seed spread close to the baseline's, for a bit more than twice the
parameters. So resnet-18 *does* carry something that a single `Linear` on `dinov2-small`
does not get out of DINOv2 alone. That is a result, and it stands with its
spread and its sigma.

**Then put it next to §4.2.** The same two points come from giving `dinov2-small` a hidden
layer instead of a second encoder, and the `dinov2-base` and `dinov2-large` trials sit
above the fused model with one encoder each. The fusion win is real; it is not the cheapest
way to that accuracy. A complete conclusion says both.

### One seed agreed this time — you could not have known

Seed 0 on its own put fusion ahead, and the seed spread confirmed it. Do not take the
wrong lesson from that. A difference of one or two points is exactly the size a single
seed can manufacture or hide, and the only way to tell a real difference from a lucky
draw is to run the other seeds. This time the win survived. The habit is the same either
way: measure the spread first, report every difference against it.

### So is fusion good or bad?

Neither, and the question is malformed. What you have measured is that *this* fusion, of
*these* two encoders, with *this* head, on *this* dataset, at *this* training size, is
worth about two points over a linear head. Change any of those and the size, or the sign,
can change with it: give both arms a hidden layer and the gap shrinks; a genuinely
complementary second stream (text alongside images, a different sensor, a different
modality) is a completely different proposition from a weaker view of the same pixels.

A student who leaves this notebook believing "fusion is good" has learned something false
just as surely as one who leaves believing "fusion is bad". What is worth keeping is the
loop:

> have an idea → state it so it could fail → build it cheaply → measure it against a
> baseline you established **first** → report what happened, including how it compares
> with the cheaper alternatives

The cache is what makes that loop cost two seconds. That is the real reason this lesson
started there.

## What you now know

- When a run costs seconds, searching beats guessing — and TPE beats grid search at it.
- Read a search one axis at a time. An axis that does not organise the scatter did not
  matter.
- On a frozen encoder the leverage is upstream: the encoder moves accuracy by more than
  ten points, the head knobs by about two. This is *not* a general law about
  architecture — see L3 and L6.
- Establish the baseline and its seed spread **before** searching, or you cannot tell a
  win from a wobble — and log it, so the comparison is one sort in MLflow.
- A result is only complete next to its alternatives: a fusion that wins by two points is
  worth less when one hidden layer, or a bigger encoder, wins by as much.

## Exercises

Notebook 05 is where you design and run your own search. Two short ones first:

**E1 — how many trials do you actually need?** Rerun the search with `N_TRIALS = 8` and
`N_TRIALS = 48`. Does the best config change? Does the *conclusion* change? Which of those
two questions is the one that matters? Compare the runs in MLflow by filtering on the
`session` tag.

**E2 — put a bottleneck in.** Project both branches to 128 before concatenating,

```python
Serial([Parallel([nn.Linear(384, 128), nn.Linear(512, 128)]), Concatenate2D(), nn.Linear(256, N_CLASSES)])
```

draw it first, the way the sum variant in §4.3 is drawn (two branches of `Block`s, a
shape on every arrow), then measure it
over seeds against `make_fusion`. Does it still beat the baseline? Explain, in one
sentence, where the information went.